In [ ]:
# # CEAM Computer Vision Challenge: Semantic Segmentation with U-Net

# SWARAJ PRASHANT SHEWALE
# 245890144

# ## Section 1: Setup
# First, ensure you have the required libraries installed.
# You can install them using pip:
# pip install tensorflow numpy matplotlib scikit-learn
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import os

print("TensorFlow version:", tf.__version__)

# ## Section 2: Configuration
# Define key parameters for the script.
IMG_HEIGHT = 128
IMG_WIDTH = 256
IMG_CHANNELS = 3
NUM_CLASSES = 5  
BATCH_SIZE = 16
EPOCHS = 20 # You can increase this for better performance, e.g., to 50 or 100

# File paths for your data
# TODO: Update this path to the directory where your .npy files are stored.
# For example: DATA_DIR = "C:/Users/sswar/Desktop/Coding_Tools/PROGRAMS/Mavro"
DATA_DIR = '.' # Use '.' to indicate the current directory
IMAGES_PATH = os.path.join(DATA_DIR, 'challenge_images.npy')
MASKS_PATH = os.path.join(DATA_DIR, 'challenge_masks.npy')

# ## Section 3: Load and Preprocess Data
print("\n--- Loading and preprocessing data... ---")
# Load the numpy arrays
try:
    images = np.load(IMAGES_PATH)
    masks = np.load(MASKS_PATH)
    print(f"Successfully loaded {len(images)} images and masks.")
except FileNotFoundError:
    print(f"Error: Make sure '{IMAGES_PATH}' and '{MASKS_PATH}' are in the same directory as the script.")
    exit()

# --- Preprocessing ---
# 1. Normalize image pixel values from [0, 255] to [0.0, 1.0]
images = images.astype(np.float32) / 255.0

# 2. Add a channel dimension to the masks for compatibility with TF/Keras
# Shape changes from (1000, 128, 256) to (1000, 128, 256, 1)
masks = np.expand_dims(masks, axis=-1)
print(f"Images shape: {images.shape}")
print(f"Masks shape: {masks.shape}")

# --- Data Splitting (70% train, 15% validation, 15% test) ---
# First, split into training (70%) and a temporary set (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    images, masks, test_size=0.30, random_state=42
)
# Then, split the temporary set into validation (15%) and test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)
print(f"\nData split:")
print(f"Training set:   {len(X_train)} samples")
print(f"Validation set: {len(X_val)} samples")
print(f"Test set:       {len(X_test)} samples")

# ## Section 4: Create TensorFlow Datasets
# Create efficient data pipelines for training, validation, and testing.
print("\n--- Creating tf.data.Dataset pipelines... ---")
def create_dataset(X, y):
    return tf.data.Dataset.from_tensor_slices((X, y))

train_dataset = create_dataset(X_train, y_train).shuffle(buffer_size=1024).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset = create_dataset(X_val, y_val).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_dataset = create_dataset(X_test, y_test).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
print("Dataset pipelines created successfully.")

# ## Section 5: Build the U-Net Segmentation Model
print("\n--- Building the U-Net model... ---")
# We define a U-Net model, a popular and effective architecture for semantic segmentation.
# It consists of an encoder (downsampling path) and a decoder (upsampling path) with skip connections.
def build_unet_model(input_shape, num_classes):
    """Builds a U-Net model using Keras functional API."""
    inputs = layers.Input(shape=input_shape)

    # --- Encoder Path ---
    # Block 1
    c1 = layers.Conv2D(16, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(inputs)
    c1 = layers.Dropout(0.1)(c1)
    c1 = layers.Conv2D(16, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c1)
    p1 = layers.MaxPooling2D((2, 2))(c1)

    # Block 2
    c2 = layers.Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(p1)
    c2 = layers.Dropout(0.1)(c2)
    c2 = layers.Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c2)
    p2 = layers.MaxPooling2D((2, 2))(c2)

    # Block 3
    c3 = layers.Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(p2)
    c3 = layers.Dropout(0.2)(c3)
    c3 = layers.Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c3)
    p3 = layers.MaxPooling2D((2, 2))(c3)

    # Block 4
    c4 = layers.Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(p3)
    c4 = layers.Dropout(0.2)(c4)
    c4 = layers.Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c4)
    p4 = layers.MaxPooling2D(pool_size=(2, 2))(c4)

    # --- Bottleneck ---
    c5 = layers.Conv2D(256, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(p4)
    c5 = layers.Dropout(0.3)(c5)
    c5 = layers.Conv2D(256, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c5)

    # --- Decoder Path ---
    # Block 6
    u6 = layers.Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(c5)
    u6 = layers.concatenate([u6, c4])
    c6 = layers.Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(u6)
    c6 = layers.Dropout(0.2)(c6)
    c6 = layers.Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c6)

    # Block 7
    u7 = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(c6)
    u7 = layers.concatenate([u7, c3])
    c7 = layers.Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(u7)
    c7 = layers.Dropout(0.2)(c7)
    c7 = layers.Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c7)

    # Block 8
    u8 = layers.Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same')(c7)
    u8 = layers.concatenate([u8, c2])
    c8 = layers.Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(u8)
    c8 = layers.Dropout(0.1)(c8)
    c8 = layers.Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c8)

    # Block 9
    u9 = layers.Conv2DTranspose(16, (2, 2), strides=(2, 2), padding='same')(c8)
    u9 = layers.concatenate([u9, c1], axis=3)
    c9 = layers.Conv2D(16, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(u9)
    c9 = layers.Dropout(0.1)(c9)
    c9 = layers.Conv2D(16, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c9)

    # Output Layer
    outputs = layers.Conv2D(num_classes, (1, 1), activation='softmax')(c9)
    model = models.Model(inputs=[inputs], outputs=[outputs])
    return model

model = build_unet_model((IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS), NUM_CLASSES)
model.summary()

# ## Section 6: Compile and Train the Model
print("\n--- Compiling the model... ---")
# We use SparseCategoricalCrossentropy because our masks are integer-encoded (not one-hot).
# MeanIoU is a standard metric for segmentation tasks.
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[tf.keras.metrics.OneHotMeanIoU(num_classes=NUM_CLASSES)],
)
print(f"\n--- Starting model training for {EPOCHS} epochs... ---")
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS
)
print("\n--- Model training finished. ---")

# ## Section 7: Plot Training History
# This section generates the graphs required by the challenge.
print("\n--- Generating training history plots... ---")
def plot_history(history):
    # Extracting the correct metric key. Keras names it based on the metric object.
    iou_metric_key = [key for key in history.history.keys() if 'one_hot_mean_io_u' in key and 'val' not in key][0]
    val_iou_metric_key = [key for key in history.history.keys() if 'one_hot_mean_io_u' in key and 'val' in key][0]
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle('Model Training History', fontsize=16)

    # Plot Training & Validation Loss
    ax1.plot(history.history['loss'], label='Training Loss', color='dodgerblue', lw=2)
    ax1.plot(history.history['val_loss'], label='Validation Loss', color='orangered', lw=2)
    ax1.set_title('Loss vs. Epochs')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend(loc='upper right')
    ax1.grid(True)

    # Plot Training & Validation IoU
    ax2.plot(history.history[iou_metric_key], label='Training IoU', color='dodgerblue', lw=2)
    ax2.plot(history.history[val_iou_metric_key], label='Validation IoU', color='orangered', lw=2)
    ax2.set_title('Mean IoU vs. Epochs')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Mean IoU')
    ax2.legend(loc='lower right')
    ax2.grid(True)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

plot_history(history)

# ## Section 8: Evaluate on Test Set
print("\n--- Evaluating model on the test set... ---")
test_loss, test_iou = model.evaluate(test_dataset)
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Mean IoU: {test_iou:.4f}")

# ## Section 9: Visualize Predictions
print("\n--- Visualizing predictions on test images... ---")
# Define a color map for visualizing the masks
colors = [
    [0, 0, 0],         # Black
    [128, 128, 128],   # Gray
    [0, 0, 255],       # Blue
    [255, 0, 0],       # Red
    [0, 128, 0]        # Green
]
colors = np.array(colors, dtype=np.uint8)

def colorize_mask(mask, num_classes):
    # Ensure mask has no channel dimension
    if mask.ndim == 3 and mask.shape[-1] == 1:
        mask = np.squeeze(mask, axis=-1)
    # Create an RGB image from the mask
    rgb_mask = colors[mask]
    return rgb_mask

def visualize_predictions(dataset, model, num_samples=5):
    plt.figure(figsize=(15, num_samples * 5))
    for i, (image, true_mask) in enumerate(dataset.take(num_samples)):
        # Predict
        pred_logits = model.predict(image)
        pred_mask = tf.argmax(pred_logits, axis=-1)

        # Plot original image
        ax = plt.subplot(num_samples, 3, i * 3 + 1)
        ax.imshow(image[0])
        ax.set_title(f"Original Image #{i+1}")
        ax.axis('off')

        # Plot ground truth mask
        ax = plt.subplot(num_samples, 3, i * 3 + 2)
        ax.imshow(colorize_mask(true_mask[0].numpy(), NUM_CLASSES))
        ax.set_title("Ground Truth Mask")
        ax.axis('off')

        # Plot predicted mask
        ax = plt.subplot(num_samples, 3, i * 3 + 3)
        ax.imshow(colorize_mask(pred_mask[0].numpy(), NUM_CLASSES))
        ax.set_title("Predicted Mask")
        ax.axis('off')
    plt.tight_layout()
    plt.show()
visualize_predictions(test_dataset, model, num_samples=5)
print("\n--- Script finished. ---")

TensorFlow version: 2.19.0

--- Loading and preprocessing data... ---
Successfully loaded 1000 images and masks.
Images shape: (1000, 128, 256, 3)
Masks shape: (1000, 128, 256, 1)

Data split:
Training set:   700 samples
Validation set: 150 samples
Test set:       150 samples

--- Creating tf.data.Dataset pipelines... ---
Dataset pipelines created successfully.

--- Building the U-Net model... ---


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 128, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_57 (Conv2D)  │ (None, 128, 256,  │        448 │ input_layer_3[0]… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_27          │ (None, 128, 256,  │          0 │ conv2d_57[0][0]   │
│ (Dropout)           │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_58 (Conv2D)  │ (None, 128, 256,  │      2,320 │ dropout_27[0][0]  │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_12    │ (None, 64, 128,   │          0 │ conv2d_58[0][0]   │
│ (MaxPooling2D)      │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_59 (Conv2D)  │ (None, 64, 128,   │      4,640 │ max_pooling2d_12… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_28          │ (None, 64, 128,   │          0 │ conv2d_59[0][0]   │
│ (Dropout)           │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_60 (Conv2D)  │ (None, 64, 128,   │      9,248 │ dropout_28[0][0]  │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_13    │ (None, 32, 64,    │          0 │ conv2d_60[0][0]   │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_61 (Conv2D)  │ (None, 32, 64,    │     18,496 │ max_pooling2d_13… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_29          │ (None, 32, 64,    │          0 │ conv2d_61[0][0]   │
│ (Dropout)           │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_62 (Conv2D)  │ (None, 32, 64,    │     36,928 │ dropout_29[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_14    │ (None, 16, 32,    │          0 │ conv2d_62[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_63 (Conv2D)  │ (None, 16, 32,    │     73,856 │ max_pooling2d_14… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_30          │ (None, 16, 32,    │          0 │ conv2d_63[0][0]   │
│ (Dropout)           │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_64 (Conv2D)  │ (None, 16, 32,    │    147,584 │ dropout_30[0][0]  │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_15    │ (None, 8, 16,     │          0 │ conv2d_64[0][0] 

 Total params: 1,941,173 (7.40 MB)

 Trainable params: 1,941,173 (7.40 MB)

 Non-trainable params: 0 (0.00 B)


--- Compiling the model... ---

--- Starting model training for 20 epochs... ---
Epoch 1/20
39/44 ━━━━━━━━━━━━━━━━━━━━ 2s 532ms/step - loss: 1.6709 - one_hot_mean_io_u_3: 0.0403